# NanoGPT v7 — 10M Parameter GPT on Google Colab GPU

Training a 10.6M parameter GPT model on the TinyStories dataset with BPE tokenization.

**What this does:**
- Downloads TinyStories (~500MB of children's stories)
- Tokenizes with GPT-2 BPE (50,257 vocab)
- Trains a transformer (n_embd=160, 8 heads, 8 layers, 256 token context)
- Generates stories!

**GPU speedup:** ~50-100x faster than CPU. 10k steps in ~15-30 min instead of 10-15 hours!

**Setup:** Runtime → Change runtime type → **T4 GPU** (free tier)

## 1. Install Dependencies & Check GPU

In [ ]:
!pip install -q tiktoken datasets

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  No GPU detected! Go to Runtime → Change runtime type → T4 GPU")

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"\nUsing device: {device}")

## 2. Download & Tokenize TinyStories with BPE

This downloads ~500MB from HuggingFace and tokenizes 2.5M stories with GPT-2's BPE tokenizer.
Takes ~5-10 minutes on Colab.

In [ ]:
import os
import pickle
import numpy as np
import tiktoken
from datasets import load_dataset

data_dir = 'tinystories_bpe'
os.makedirs(data_dir, exist_ok=True)

# Skip if already prepared
if os.path.exists(os.path.join(data_dir, 'train.bin')):
    print("Data already prepared! Skipping download.")
else:
    print("Loading TinyStories dataset from HuggingFace...")
    dataset = load_dataset("roneneldan/TinyStories")
    print(f"Train stories: {len(dataset['train']):,}")
    print(f"Validation stories: {len(dataset['validation']):,}")

    # GPT-2 BPE tokenizer
    enc = tiktoken.get_encoding("gpt2")
    vocab_size = enc.n_vocab  # 50,257
    print(f"BPE vocabulary size: {vocab_size:,}")

    # Tokenize and write to disk in CHUNKS to avoid OOM!
    # Why? 472M Python ints × 28 bytes each ≈ 13GB RAM.
    # Colab free tier only has ~12GB. So we stream to disk instead.
    CHUNK_SIZE = 50_000  # stories per chunk

    def tokenize_to_file(split, filepath):
        """Tokenize a dataset split and stream to disk in chunks."""
        data = dataset[split]
        n_stories = len(data)
        total_tokens = 0

        with open(filepath, 'wb') as f:
            for start in range(0, n_stories, CHUNK_SIZE):
                end = min(start + CHUNK_SIZE, n_stories)
                chunk_ids = []
                for i in range(start, end):
                    chunk_ids.extend(enc.encode_ordinary(data[i]['text']))

                # Convert chunk to uint16 and write immediately, then free memory
                chunk_arr = np.array(chunk_ids, dtype=np.uint16)
                chunk_arr.tofile(f)
                total_tokens += len(chunk_ids)

                print(f"  {end:,} / {n_stories:,} stories ({total_tokens:,} tokens)")

        return total_tokens

    print("\nTokenizing training data (streaming to disk)...")
    n_train = tokenize_to_file('train', os.path.join(data_dir, 'train.bin'))
    print(f"Train BPE tokens: {n_train:,}")

    print("\nTokenizing validation data...")
    n_val = tokenize_to_file('validation', os.path.join(data_dir, 'val.bin'))
    print(f"Val BPE tokens: {n_val:,}")

    meta = {'vocab_size': vocab_size, 'tokenizer': 'gpt2'}
    with open(os.path.join(data_dir, 'meta.pkl'), 'wb') as f:
        pickle.dump(meta, f)

    print(f"\nSaved to {data_dir}/")
    print(f"  train.bin: {os.path.getsize(os.path.join(data_dir, 'train.bin')) / 1e6:.1f} MB")
    print(f"  val.bin: {os.path.getsize(os.path.join(data_dir, 'val.bin')) / 1e6:.1f} MB")

## 3. Define the GPT Model (v7 — 10.6M params)

Architecture: token + position embeddings → 8 transformer blocks → output projection

| Config | Value |
|--------|-------|
| n_embd | 160 |
| n_head | 8 (head_size=20) |
| n_layer | 8 |
| block_size | 256 BPE tokens (~1024 chars) |
| vocab_size | 50,257 (GPT-2 BPE) |
| dropout | 0.1 |
| Weight tying | Yes |
| Init | GPT-2 style (std=0.02) |
| Total params | 10,552,800 |

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(1337)

# Hyperparameters
block_size = 256
n_embd = 160
n_head = 8
n_layer = 8
dropout = 0.1


class Head(nn.Module):
    """Single head of self-attention."""
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2, -1) * (k.shape[-1] ** -0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)
        return wei @ v


class MultiHeadAttention(nn.Module):
    """Multiple heads of self-attention in parallel."""
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))


class FeedForward(nn.Module):
    """Feed-forward network with GELU."""
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    """Transformer block: attention + feed-forward with residual connections."""
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x


class GPTLanguageModel(nn.Module):
    """v7: 10M parameter GPT with BPE tokenization."""
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)
        self.lm_head.weight = self.token_embedding_table.weight  # weight tying
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=idx.device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        if targets is None:
            return logits, None
        B, T, C = logits.shape
        return logits.view(B*T, C), F.cross_entropy(logits.view(B*T, C), targets.view(B*T))

    def generate(self, idx, max_new_tokens, temperature=0.8, top_k=40):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = float('-inf')
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx


# Print model info
vocab_size = 50257
model = GPTLanguageModel(vocab_size).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"GPT v7: {n_params:,} parameters ({n_params/1e6:.1f}M)")
print(f"  Token embeddings:  {vocab_size * n_embd:,} ({vocab_size * n_embd / n_params * 100:.0f}%)")
print(f"  Transformer blocks: {n_params - vocab_size * n_embd - block_size * n_embd:,} ({(n_params - vocab_size * n_embd - block_size * n_embd) / n_params * 100:.0f}%)")
print(f"  Context: {block_size} BPE tokens ≈ {block_size * 4} chars")
print(f"  Device: {device}")

## 4. Train!

On a T4 GPU, 10k steps should take ~15-30 minutes (vs 10-15 hours on CPU).

**If you hit OOM:** Runtime → Restart runtime, then re-run all cells from the top.

In [ ]:
import time
import math

# Load data
with open(os.path.join(data_dir, 'meta.pkl'), 'rb') as f:
    meta = pickle.load(f)
vocab_size = meta['vocab_size']

train_data = np.memmap(os.path.join(data_dir, 'train.bin'), dtype=np.uint16, mode='r')
val_data = np.memmap(os.path.join(data_dir, 'val.bin'), dtype=np.uint16, mode='r')
print(f"Train tokens: {len(train_data):,}")
print(f"Val tokens: {len(val_data):,}")
print(f"Tokens per parameter: {len(train_data) / n_params:.0f}x")

# Training hyperparameters
# batch_size=32 fits T4 (15GB VRAM) in fp32. If you get OOM, restart runtime first.
batch_size = 32 if device == 'cuda' else 16
eval_iters = 200
eval_interval = 500
max_iters = 10_000

# Learning rate schedule
max_lr = 3e-4
min_lr = max_lr * 0.1
warmup_steps = 500


def get_lr(step):
    if step < warmup_steps:
        return max_lr * (step + 1) / warmup_steps
    decay_steps = max_iters - warmup_steps
    step_in_decay = step - warmup_steps
    cosine_decay = 0.5 * (1 + math.cos(math.pi * step_in_decay / decay_steps))
    return min_lr + (max_lr - min_lr) * cosine_decay


def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([torch.from_numpy(data[i:i+block_size].astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy(data[i+1:i+block_size+1].astype(np.int64)) for i in ix])
    return x.to(device), y.to(device)


@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out


optimizer = torch.optim.AdamW(model.parameters(), lr=max_lr, weight_decay=1e-2)

print(f"\nTraining v7 ({n_params/1e6:.1f}M params) on TinyStories BPE")
print(f"  batch_size={batch_size}, max_iters={max_iters:,}")
print(f"  LR: warmup {warmup_steps} steps → cosine decay")
print(f"  Device: {device}")
print()

start_time = time.time()

for step in range(max_iters):
    lr = get_lr(step)
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr

    if step % eval_interval == 0 or step == max_iters - 1:
        losses = estimate_loss()
        elapsed = time.time() - start_time
        print(f"step {step:5d}: train {losses['train']:.4f}, val {losses['val']:.4f}, lr {lr:.2e} [{elapsed:.0f}s]")

    xb, yb = get_batch('train')
    _, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

total_time = time.time() - start_time
print(f"\nDone! Total time: {total_time:.0f}s ({total_time/60:.1f} min)")

## 5. Generate Stories!

In [ ]:
enc = tiktoken.get_encoding("gpt2")

model.eval()
for i in range(3):
    context = torch.zeros((1, 1), dtype=torch.long, device=device)
    generated_ids = model.generate(context, max_new_tokens=200)[0].tolist()
    print(f"--- Story {i+1} ---")
    print(enc.decode(generated_ids))
    print()

## 6. Generate from a Prompt (optional)

In [ ]:
prompt = "Once upon a time, there was a little"
prompt_ids = enc.encode(prompt)
context = torch.tensor([prompt_ids], dtype=torch.long, device=device)

model.eval()
generated_ids = model.generate(context, max_new_tokens=200, temperature=0.8, top_k=40)[0].tolist()
print(enc.decode(generated_ids))

## 7. Save/Download Model (optional)

Save the trained model so you can download it or load it later.

In [ ]:
# Save model checkpoint
checkpoint = {
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'config': {
        'vocab_size': vocab_size,
        'n_embd': n_embd,
        'n_head': n_head,
        'n_layer': n_layer,
        'block_size': block_size,
        'dropout': dropout,
    },
    'train_loss': losses['train'].item(),
    'val_loss': losses['val'].item(),
}
torch.save(checkpoint, 'nanogpt_v7_tinystories.pt')
size_mb = os.path.getsize('nanogpt_v7_tinystories.pt') / 1e6
print(f"Saved checkpoint: nanogpt_v7_tinystories.pt ({size_mb:.1f} MB)")
print("Download from Colab: Files tab (left sidebar) → right-click → Download")